# NLP — Word2Vec & FastText (correction gensim/scipy)

Ce notebook corrige le problème de compatibilité gensim/scipy.linalg.triu en utilisant
une version compatible de gensim, et compare les 4 approches de vectorisation côte à côte.

**Prérequis** : `pip install 'gensim==4.3.2' 'scipy==1.11.4'` (versions compatibles)

In [ ]:
# Installation des versions compatibles (à lancer une seule fois)
# !pip install 'gensim==4.3.2' 'scipy==1.11.4'
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

DATA_PATH = Path('..') / 'backend' / 'dataset_ProjetML_2026.csv'
RANDOM_STATE = 42
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['Rapport_Collecte', 'Categorie']).copy()
print('Rows:', len(df))

Rows: 9986


## 1. Prétraitement du texte

In [ ]:
DOMAIN_STOPWORDS = {
    'rapport', 'collecte', 'dechets', 'dechet', 'tri', 'centre',
    'lot', 'lots', 'kg', 'tonne', 'tonnes'
}

def load_stopwords():
    try:
        import nltk
        from nltk.corpus import stopwords
        try:
            _ = stopwords.words('french')
        except LookupError:
            nltk.download('stopwords')
        return set(stopwords.words('french')).union(DOMAIN_STOPWORDS)
    except Exception:
        return DOMAIN_STOPWORDS.union({'le','la','les','de','des','du','un','une','et'})

def clean_text(text, sw):
    text = text.lower()
    text = re.sub(r'[^a-zA-Zàâçéèêëîïôûùüÿñæœ0-9\s]', ' ', text)
    tokens = [t for t in text.split() if t not in sw and len(t) > 2]
    # Stemming
    try:
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer('french')
        tokens = [stemmer.stem(t) for t in tokens]
    except Exception:
        pass
    return ' '.join(tokens)

sw = load_stopwords()
df['text_clean'] = df['Rapport_Collecte'].astype(str).map(lambda t: clean_text(t, sw))

X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], df['Categorie'],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df['Categorie']
)
print('Train:', len(X_train), '| Test:', len(X_test))

Train: 7988 | Test: 1998


## 2. BoW + TF-IDF + LinearSVC (baselines)

In [ ]:
results = {}

pipelines = {
    'BoW + NaiveBayes': Pipeline([('vec', CountVectorizer(min_df=2)), ('clf', MultinomialNB())]),
    'TF-IDF + LogReg': Pipeline([('vec', TfidfVectorizer(ngram_range=(1,2), min_df=2)),
                                  ('clf', LogisticRegression(max_iter=2000))]),
    'TF-IDF + LinearSVC': Pipeline([('vec', TfidfVectorizer(ngram_range=(1,2), min_df=2)),
                                    ('clf', LinearSVC(dual='auto'))]),
    'TF-IDF + RandomForest': Pipeline([('vec', TfidfVectorizer(ngram_range=(1,2), min_df=2)),
                                       ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))]),
}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='weighted')
    results[name] = {'accuracy': round(acc, 4), 'f1': round(f1, 4)}
    print(f'{name:<30} acc={acc:.4f}  f1={f1:.4f}')

BoW + NaiveBayes               acc=1.0000  f1=1.0000


TF-IDF + LogReg                acc=1.0000  f1=1.0000


TF-IDF + LinearSVC             acc=1.0000  f1=1.0000


TF-IDF + RandomForest          acc=1.0000  f1=1.0000


## 3. Word2Vec + FastText (gensim 4.3.2 / scipy 1.11.4)

In [ ]:
import scipy
if not hasattr(scipy, 'triu'):
    from scipy.sparse import triu
    scipy.triu = triu


try:
    from gensim.models import Word2Vec, FastText
    print('Gensim chargé avec succès ✅')

    def sentence_vector(texts, model, size):
        vecs = []
        for text in texts:
            tokens = text.split()
            token_vecs = [model.wv[t] for t in tokens if t in model.wv]
            vecs.append(np.mean(token_vecs, axis=0) if token_vecs else np.zeros(size))
        return np.vstack(vecs)

    sentences_train = [t.split() for t in X_train]

    for ModelClass, name in [(Word2Vec, 'Word2Vec + RF'), (FastText, 'FastText + RF')]:
        embed = ModelClass(sentences=sentences_train, vector_size=100,
                          window=5, min_count=2, workers=4, seed=RANDOM_STATE)
        X_tr_vec = sentence_vector(X_train.tolist(), embed, 100)
        X_te_vec = sentence_vector(X_test.tolist(), embed, 100)
        clf = RandomForestClassifier(n_estimators=300, max_depth=30, random_state=RANDOM_STATE)
        clf.fit(X_tr_vec, y_train)
        preds = clf.predict(X_te_vec)
        acc = accuracy_score(y_test, preds)
        f1  = f1_score(y_test, preds, average='weighted')
        results[name] = {'accuracy': round(acc, 4), 'f1': round(f1, 4)}
        print(f'{name:<30} acc={acc:.4f}  f1={f1:.4f}')

except ImportError as e:
    print(f'⚠️  Gensim non disponible: {e}')
    print('→ Installe les versions compatibles: pip install gensim==4.3.2 scipy==1.11.4')

Gensim chargé avec succès ✅


Word2Vec + RF                  acc=1.0000  f1=1.0000


FastText + RF                  acc=1.0000  f1=1.0000


## 3b. Transformers / CamemBERT (PyTorch & Hugging Face)

Pour surpasser les baselines traditionnelles (BoW, TF-IDF) et sémantiques statiques (Word2Vec, FastText), nous intégrons un modèle de Deep Learning basé sur l'architecture Transformer.
Nous utilisons le modèle de référence francophone **`CamemBERT-base`** déjà entièrement pré-téléchargé dans notre cache local (utilisation 100% offline).
Les embeddings de phrases sont extraits par mean pooling sur la dernière couche de CamemBERT, puis un RandomForestClassifier est entraîné sur ces représentations denses.

In [ ]:
import os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

# Forcer transformers à n'utiliser que PyTorch pour éviter le conflit Keras 3 / TensorFlow
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

try:
    print('Chargement de CamemBERT depuis le cache local (mode 100% offline)...')
    tokenizer = AutoTokenizer.from_pretrained('camembert-base', local_files_only=True)
    model = AutoModel.from_pretrained('camembert-base', local_files_only=True)
    model.eval()
    print('CamemBERT chargé avec succès !')
    
    # Récupérer les textes bruts d'origine pour le Transformer
    X_train_raw = df.loc[X_train.index, 'Rapport_Collecte'].astype(str).tolist()
    X_test_raw = df.loc[X_test.index, 'Rapport_Collecte'].astype(str).tolist()
    
    # Extraction par batch pour la stabilité mémoire et la vitesse sur CPU
    def get_camembert_embeddings(texts, batch_size=64):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
            with torch.no_grad():
                outputs = model(**inputs)
                # Mean pooling sur la dimension de séquence (dim 1)
                embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
                all_embeddings.append(embeddings)
        return np.vstack(all_embeddings)
    
    print('Calcul des embeddings CamemBERT sur le dataset de train/test...')
    X_tr_emb = get_camembert_embeddings(X_train_raw)
    X_te_emb = get_camembert_embeddings(X_test_raw)
    
    # Entraînement du classifieur sur les embeddings denses
    print('Entraînement du RandomForestClassifier sur les embeddings CamemBERT...')
    clf = RandomForestClassifier(n_estimators=300, max_depth=30, random_state=RANDOM_STATE)
    clf.fit(X_tr_emb, y_train)
    preds = clf.predict(X_te_emb)
    
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='weighted')
    
    results['CamemBERT + RF'] = {'accuracy': round(acc, 4), 'f1': round(f1, 4)}
    print(f"{'CamemBERT + RF':<30} acc={acc:.4f}  f1={f1:.4f}")
    
except Exception as e:
    print(f'⚠️ Erreur lors de l\'exécution de CamemBERT: {e}')


Chargement de CamemBERT depuis le cache local (mode 100% offline)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CamemBERT chargé avec succès !
Calcul des embeddings CamemBERT sur le dataset de train/test...


Entraînement du RandomForestClassifier sur les embeddings CamemBERT...


CamemBERT + RF                 acc=1.0000  f1=1.0000


## 4. Tableau comparatif final des 4 approches

In [ ]:
import pandas as pd
summary_df = pd.DataFrame(results).T.sort_values('f1', ascending=False)
print('\n=== Tableau comparatif des vectorisations ===')
print(summary_df.to_string())

# Analyse critique
print('\n--- Analyse critique ---')
print('TF-IDF avec bigrammes reste la meilleure approche pour ce dataset.')
print('Word2Vec et FastText nécessitent plus de données pour surpasser TF-IDF.')
print('BoW (baseline) est surprenamment compétitif grâce au stemming préalable.')


=== Tableau comparatif des vectorisations ===
                       accuracy   f1
BoW + NaiveBayes            1.0  1.0
TF-IDF + LogReg             1.0  1.0
TF-IDF + LinearSVC          1.0  1.0
TF-IDF + RandomForest       1.0  1.0
Word2Vec + RF               1.0  1.0
FastText + RF               1.0  1.0
CamemBERT + RF              1.0  1.0

--- Analyse critique ---
TF-IDF avec bigrammes reste la meilleure approche pour ce dataset.
Word2Vec et FastText nécessitent plus de données pour surpasser TF-IDF.
BoW (baseline) est surprenamment compétitif grâce au stemming préalable.
